In [ ]:
using System;
using System.Collections.Generic;
using System.Collections.ObjectModel;
using System.Linq;

namespace InvoiceSystem
{
    public class LineItem
    {
        public string Description { get; set; }
        public decimal Quantity { get; set; }
        public decimal UnitPrice { get; set; }
        
        public decimal Total => Quantity * UnitPrice;

        public DateTime? SupplyDate { get; set; }
        public string CancellationReason { get; set; } = string.Empty;
        public bool IsReturned { get; set; }

        public LineItem(string description, decimal quantity, decimal unitPrice)
        {
            Description = description;
            Quantity = quantity;
            UnitPrice = unitPrice;
        }
    }

    public class Invoice
    {
        public string InvoiceNumber { get; protected set; }
        public DateTime IssueDate { get; protected set; }
        public decimal TotalAmount { get; protected set; }

        protected List<LineItem> LineItems { get; } = new List<LineItem>();

        public Invoice(string invoiceNumber, DateTime issueDate)
        {
            InvoiceNumber = invoiceNumber;
            IssueDate = issueDate;
        }

        public virtual void CalculateTotal()
        {
            TotalAmount = LineItems.Sum(item => item.Total);
        }

        public virtual void AddLine(LineItem lineItem)
        {
            LineItems.Add(lineItem);
            CalculateTotal();
        }

        public virtual void RemoveLine(LineItem lineItem)
        {
            LineItems.Remove(lineItem);
            CalculateTotal();
        }

        public ReadOnlyCollection<LineItem> GetLineItems()
        {
            return new ReadOnlyCollection<LineItem>(LineItems);
        }

        public void PrintInfo()
        {
            Console.WriteLine($"Invoice #{InvoiceNumber} | Date: {IssueDate:dd.MM.yyyy} | Total: {TotalAmount:C}");
        }
    }

    public class GoodsInvoice : Invoice
    {
        public DateTime SupplyDate { get; set; }

        public GoodsInvoice(string invoiceNumber, DateTime issueDate, DateTime supplyDate)
            : base(invoiceNumber, issueDate)
        {
            SupplyDate = supplyDate;
        }

        public override void AddLine(LineItem lineItem)
        {
            lineItem.SupplyDate = SupplyDate;
            base.AddLine(lineItem);
            Console.WriteLine($"[GoodsInvoice] Item added: {lineItem.Description}. Supply date set: {SupplyDate:dd.MM.yyyy}");
        }
    }

    public class ServiceInvoice : Invoice
    {
        public DateTime ServiceDate { get; set; }
        
        private readonly Dictionary<LineItem, string> _cancellationLog = new Dictionary<LineItem, string>();

        public string PendingCancellationReason { get; set; } = string.Empty;

        public ServiceInvoice(string invoiceNumber, DateTime issueDate, DateTime serviceDate)
            : base(invoiceNumber, issueDate)
        {
            ServiceDate = serviceDate;
        }

        public override void RemoveLine(LineItem lineItem)
        {
            if (string.IsNullOrWhiteSpace(PendingCancellationReason))
                PendingCancellationReason = "Reason not specified";

            _cancellationLog[lineItem] = PendingCancellationReason;
            lineItem.CancellationReason = PendingCancellationReason;
            
            base.RemoveLine(lineItem);
            Console.WriteLine($"[ServiceInvoice] Service removed: {lineItem.Description}. Reason: {PendingCancellationReason}");
            PendingCancellationReason = string.Empty;
        }

        public void PrintCancellationLog()
        {
            Console.WriteLine("Cancellation Log:");
            foreach (var entry in _cancellationLog)
                Console.WriteLine($"  - {entry.Key.Description}: {entry.Value}");
        }
    }

    public class CombinedInvoice : Invoice
    {
        public bool ReturnAllowed { get; set; }

        public CombinedInvoice(string invoiceNumber, DateTime issueDate, bool returnAllowed)
            : base(invoiceNumber, issueDate)
        {
            ReturnAllowed = returnAllowed;
        }

        public override void CalculateTotal()
        {
            if (!ReturnAllowed)
            {
                base.CalculateTotal();
                Console.WriteLine("[CombinedInvoice] Returns disabled. Standard calculation applied.");
                return;
            }

            decimal fullAmount = LineItems.Sum(item => item.Total);
            decimal returnedAmount = LineItems.Where(item => item.IsReturned).Sum(item => item.Total);
            TotalAmount = fullAmount - returnedAmount;

            Console.WriteLine($"[CombinedInvoice] Returns allowed. Adjusted total: {TotalAmount:C} (Returned: {returnedAmount:C})");
        }
    }

    public class InvoiceProcessor
    {
        public void ProcessBatch(IEnumerable<Invoice> invoices)
        {
            foreach (var invoice in invoices)
            {
                invoice.CalculateTotal();
                invoice.PrintInfo();
            }
        }

        public bool TransferLineItem(Invoice source, Invoice destination, LineItem item)
        {
            if (!source.GetLineItems().Contains(item))
            {
                Console.WriteLine($"[Processor] Item '{item.Description}' not found in source invoice {source.InvoiceNumber}.");
                return false;
            }

            source.RemoveLine(item);
            destination.AddLine(item);
            Console.WriteLine($"[Processor] Transferred '{item.Description}' from {source.InvoiceNumber} to {destination.InvoiceNumber}");
            return true;
        }

        public decimal CalculateGrandTotal(IEnumerable<Invoice> invoices)
        {
            return invoices.Sum(inv => inv.TotalAmount);
        }
    }

    class Program
    {
        static void Main()
        {
            Console.WriteLine("--- 1. Создание объектов через конструкторы и сеттеры ---");
            
            var goodsInv = new GoodsInvoice("G-001", new DateTime(2026, 4, 20), new DateTime(2026, 4, 25));
            var serviceInv = new ServiceInvoice("S-002", new DateTime(2026, 4, 20), new DateTime(2026, 4, 22));
            var combinedInv = new CombinedInvoice("C-003", new DateTime(2026, 4, 20), true);

            goodsInv.SupplyDate = new DateTime(2026, 4, 26);
            serviceInv.PendingCancellationReason = "Client requested early termination";

            Console.WriteLine("\n--- 2. Добавление позиций (полиморфный вызов AddLine) ---");
            var laptop = new LineItem("Laptop Pro", 2, 1500m);
            var consulting = new LineItem("IT Consulting", 10, 200m);
            var cables = new LineItem("Network Cables", 50, 10m);
            var monitor = new LineItem("Monitor 27", 1, 400m) { IsReturned = true };

            goodsInv.AddLine(laptop);
            serviceInv.AddLine(consulting);
            combinedInv.AddLine(cables);
            combinedInv.AddLine(monitor);

            Console.WriteLine("\n--- 3. Взаимодействие объектов через InvoiceProcessor ---");
            var processor = new InvoiceProcessor();

            List<Invoice> batch = new List<Invoice> { goodsInv, serviceInv, combinedInv };
            processor.ProcessBatch(batch);

            Console.WriteLine("\n--- 4. Передача позиции из GoodsInvoice в CombinedInvoice ---");
            processor.TransferLineItem(goodsInv, combinedInv, laptop);

            Console.WriteLine("\n--- 5. Финальные итоги ---");
            goodsInv.PrintInfo();
            serviceInv.PrintInfo();
            combinedInv.PrintInfo();
            serviceInv.PrintCancellationLog();

            decimal grandTotal = processor.CalculateGrandTotal(batch);
            Console.WriteLine($"\nGrand Total (all invoices): {grandTotal:C}");

            Console.WriteLine("\n--- 6. Прямая демонстрация полиморфизма ---");
            foreach (var inv in batch)
            {
                Console.WriteLine($"Processing {inv.GetType().Name}...");
                inv.CalculateTotal();
            }
        }
    }
}